<a href="https://colab.research.google.com/github/ShilpaShyamal/Bankmarketingproject/blob/main/Smote1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

 # INSTALL (FIRST TIME ONLY)

!pip install xgboost lightgbm catboost imbalanced-learn


# IMPORT

import numpy as np
import pandas as pd

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import *
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, BaggingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from imblearn.over_sampling import SMOTE
from imblearn.metrics import geometric_mean_score


# DATASET

from sklearn.datasets import make_classification
X, y = make_classification(n_samples=1000, n_features=20,
                           weights=[0.7, 0.3], random_state=42)


# 10 FOLD

kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# METRICS FUNCTION

def get_metrics(y_true, y_pred, y_prob):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    specificity = tn/(tn+fp)

    return [
        accuracy_score(y_true, y_pred),
        precision_score(y_true, y_pred),
        recall_score(y_true, y_pred),
        specificity,
        f1_score(y_true, y_pred),
        roc_auc_score(y_true, y_prob),
        matthews_corrcoef(y_true, y_pred),
        geometric_mean_score(y_true, y_pred),
        cohen_kappa_score(y_true, y_pred)
    ]

columns = ["Accuracy","Precision","Recall","Specificity","F1",
           "ROC-AUC","MCC","G-Mean","Kappa"]

# MODELS + PARAM

models = {
"Logistic Regression": (LogisticRegression(max_iter=1000), {"C":[0.1,1,10]}),
"Decision Tree": (DecisionTreeClassifier(), {"max_depth":[3,5,10]}),
"Random Forest": (RandomForestClassifier(), {"n_estimators":[100,200]}),
"Gradient Boosting": (GradientBoostingClassifier(), {"n_estimators":[100]}),
"XGBoost": (XGBClassifier(use_label_encoder=False, eval_metric='logloss'), {"n_estimators":[100]}),
"LightGBM": (LGBMClassifier(), {"n_estimators":[100]}),
"CatBoost": (CatBoostClassifier(verbose=0), {"iterations":[100]}),
"SVM": (SVC(probability=True), {"C":[1]}),
"KNN": (KNeighborsClassifier(), {"n_neighbors":[5]}),
"MLP": (MLPClassifier(max_iter=500), {"hidden_layer_sizes":[(50,)]}),
"Bagging": (BaggingClassifier(), {"n_estimators":[50]}),
"Stacking": (StackingClassifier(
    estimators=[('rf', RandomForestClassifier()), ('dt', DecisionTreeClassifier())],
    final_estimator=LogisticRegression()), {"final_estimator__C":[1]})
}


# MAIN LOOP

final_results = []

for name, (model, params) in models.items():

    print(f"\n=========== {name} ===========")

    search = RandomizedSearchCV(model, params, n_iter=2, cv=10, n_jobs=-1)
    search.fit(X, y)
    best_model = search.best_estimator_

    fold_data = []

    for fold, (train_idx, test_idx) in enumerate(kfold.split(X, y),1):

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # SMOTE
        sm = SMOTE(random_state=42)
        X_train, y_train = sm.fit_resample(X_train, y_train)

        best_model.fit(X_train, y_train)

        y_pred = best_model.predict(X_test)

        if hasattr(best_model,"predict_proba"):
            y_prob = best_model.predict_proba(X_test)[:,1]
        else:
            y_prob = y_pred

        row = get_metrics(y_test, y_pred, y_prob)
        fold_data.append(row)

    # Fold-wise DataFrame
    df = pd.DataFrame(fold_data, columns=columns)
    df.index = [f"Fold {i}" for i in range(1,11)]

    print("\nFold-wise Result:")
    print(df)

    # Average
    avg = df.mean()
    print("\nAverage:")
    print(avg)

    avg["Model"] = name
    final_results.append(avg)


# FINAL TABLE

final_df = pd.DataFrame(final_results)
final_df = final_df.set_index("Model")

print("\n=========== FINAL RESULT ===========")
print(final_df)

# SAVE
final_df.to_csv("/content/bank-additional-full (1) (1) (1).csv")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.6 MB/s eta 0:00:00

=========== Logistic Regression ===========

Fold-wise Result:
         Accuracy  Precision    Recall  Specificity        F1   ROC-AUC  \
Fold 1       0.89   0.794118  0.870968     0.898551  0.830769  0.917719   
Fold 2       0.88   0.764706  0.866667     0.885714  0.812500  0.956190   
Fold 3       0.88   0.875000  0.700000     0.957143  0.777778  0.907143   
Fold 4       0.93   0.828571  0.966667     0.914286  0.892308  0.971429   
Fold 5       0.84   0.750000  0.700000     0.900000  0.724138  0.909524   
Fold 6       0.89   0.787879  0.866667     0.900000  0.825397  0.938095   
Fold 7       0.90   0.833333  0.833333     0.928571  0.833333  0.888095   
Fold 8       0.85   0.692308  0.900000     0.828571  0.782609  0.934286   
Fold 9       0.88   0.764706  0.866667     0.885714  0.812500  0.943333   
Fold 10      0.86   0.700000  0.933333     0.828571  0.800000  0.932857   

              MCC    G-Mean     

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 1 is smaller than n_iter=2. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(



Fold-wise Result:
         Accuracy  Precision    Recall  Specificity        F1   ROC-AUC  \
Fold 1       0.91   0.843750  0.870968     0.927536  0.857143  0.918654   
Fold 2       0.93   0.848485  0.933333     0.928571  0.888889  0.987143   
Fold 3       0.91   0.956522  0.733333     0.985714  0.830189  0.904762   
Fold 4       0.95   0.857143  1.000000     0.928571  0.923077  0.982857   
Fold 5       0.88   0.821429  0.766667     0.928571  0.793103  0.926667   
Fold 6       0.92   0.866667  0.866667     0.942857  0.866667  0.953333   
Fold 7       0.90   0.833333  0.833333     0.928571  0.833333  0.895714   
Fold 8       0.91   0.818182  0.900000     0.914286  0.857143  0.980000   
Fold 9       0.88   0.764706  0.866667     0.885714  0.812500  0.979048   
Fold 10      0.89   0.771429  0.900000     0.885714  0.830769  0.965238   

              MCC    G-Mean     Kappa  
Fold 1   0.791686  0.898807  0.791474  
Fold 2   0.839991  0.930949  0.837963  
Fold 3   0.782993  0.850210  0.7704

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 1 is smaller than n_iter=2. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [10:49:51] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [10:49:51] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [10:49:51] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:20


Fold-wise Result:
         Accuracy  Precision    Recall  Specificity        F1   ROC-AUC  \
Fold 1       0.93   0.900000  0.870968     0.956522  0.885246  0.928004   
Fold 2       0.92   0.923077  0.800000     0.971429  0.857143  0.972381   
Fold 3       0.91   1.000000  0.700000     1.000000  0.823529  0.903810   
Fold 4       0.96   0.882353  1.000000     0.942857  0.937500  0.983333   
Fold 5       0.93   0.925926  0.833333     0.971429  0.877193  0.921905   
Fold 6       0.90   0.857143  0.800000     0.942857  0.827586  0.948571   
Fold 7       0.91   0.862069  0.833333     0.942857  0.847458  0.911429   
Fold 8       0.91   0.800000  0.933333     0.900000  0.861538  0.981429   
Fold 9       0.94   0.928571  0.866667     0.971429  0.896552  0.976667   
Fold 10      0.92   0.843750  0.900000     0.928571  0.870968  0.964286   

              MCC    G-Mean     Kappa  
Fold 1   0.835138  0.912743  0.834906  
Fold 2   0.805940  0.881557  0.801980  
Fold 3   0.787562  0.836660  0.7656

...


Fold-wise Result:
         Accuracy  Precision    Recall  Specificity        F1   ROC-AUC  \
Fold 1       0.81   0.666667  0.774194     0.826087  0.716418  0.913043   
Fold 2       0.77   0.585366  0.800000     0.757143  0.676056  0.894048   
Fold 3       0.82   0.687500  0.733333     0.857143  0.709677  0.848333   
Fold 4       0.80   0.604167  0.966667     0.728571  0.743590  0.939286   
Fold 5       0.82   0.666667  0.800000     0.828571  0.727273  0.866429   
Fold 6       0.88   0.736842  0.933333     0.857143  0.823529  0.928333   
Fold 7       0.79   0.609756  0.833333     0.771429  0.704225  0.874048   
Fold 8       0.76   0.571429  0.800000     0.742857  0.666667  0.855238   
Fold 9       0.80   0.619048  0.866667     0.771429  0.722222  0.900952   
Fold 10      0.77   0.577778  0.866667     0.728571  0.693333  0.871905   

              MCC    G-Mean     Kappa  
Fold 1   0.578387  0.799719  0.574754  
Fold 2   0.519109  0.778276  0.504310  
Fold 3   0.580073  0.792825  0.5794

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 1 is smaller than n_iter=2. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perce


Fold-wise Result:
         Accuracy  Precision    Recall  Specificity        F1   ROC-AUC  \
Fold 1       0.86   0.793103  0.741935     0.913043  0.766667  0.889201   
Fold 2       0.82   0.700000  0.700000     0.871429  0.700000  0.881905   
Fold 3       0.82   0.800000  0.533333     0.942857  0.640000  0.863333   
Fold 4       0.85   0.714286  0.833333     0.857143  0.769231  0.932857   
Fold 5       0.81   0.666667  0.733333     0.842857  0.698413  0.859048   
Fold 6       0.88   0.846154  0.733333     0.942857  0.785714  0.911429   
Fold 7       0.82   0.750000  0.600000     0.914286  0.666667  0.854762   
Fold 8       0.81   0.648649  0.800000     0.814286  0.716418  0.902381   
Fold 9       0.85   0.758621  0.733333     0.900000  0.745763  0.914286   
Fold 10      0.80   0.647059  0.733333     0.828571  0.687500  0.875238   

              MCC    G-Mean     Kappa  
Fold 1   0.667582  0.823055  0.666825  
Fold 2   0.571429  0.781025  0.571429  
Fold 3   0.545545  0.709124  0.5263

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 1 is smaller than n_iter=2. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(



Fold-wise Result:
         Accuracy  Precision    Recall  Specificity        F1   ROC-AUC  \
Fold 1       0.90   0.818182  0.870968     0.913043  0.843750  0.923562   
Fold 2       0.90   0.812500  0.866667     0.914286  0.838710  0.976429   
Fold 3       0.91   0.956522  0.733333     0.985714  0.830189  0.910714   
Fold 4       0.97   0.909091  1.000000     0.957143  0.952381  0.991667   
Fold 5       0.88   0.821429  0.766667     0.928571  0.793103  0.910952   
Fold 6       0.91   0.838710  0.866667     0.928571  0.852459  0.944762   
Fold 7       0.90   0.833333  0.833333     0.928571  0.833333  0.915238   
Fold 8       0.89   0.756757  0.933333     0.871429  0.835821  0.957143   
Fold 9       0.92   0.866667  0.866667     0.942857  0.866667  0.954286   
Fold 10      0.92   0.823529  0.933333     0.914286  0.875000  0.974048   

              MCC    G-Mean     Kappa  
Fold 1   0.771140  0.891757  0.770326  
Fold 2   0.767193  0.890158  0.766355  
Fold 3   0.782993  0.850210  0.7704